# Labelling Pipeline

This notebook implements the labelling strategy used to build training data for our NLP benchmark classifier. The pipeline operates in three phases:

1. **RegEx bucketing** — Heuristic scoring assigns each paper to one of three confidence buckets (A, B, C) based on lexical signals in the title and abstract.
2. **Stratified sampling** — Papers are drawn from each bucket to form balanced gold, train, and validation splits.
3. **LLM labelling** — Two LLMs (Mistral Small, Claude Haiku) independently classify sampled papers; inter-annotator agreement is measured with Cohen's $\kappa$.

## 1. Configuration & Data Loading

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
DATA_DIR = Path("../data")

# Sampling parameters
SEED = 42
GOLD_PER_BUCKET = 70
TRAIN_N_A, TRAIN_N_B, TRAIN_N_C = 500, 400, 300
VAL_FRAC = 0.15

In [4]:
anthology_df = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")
print(f"Anthology corpus: {len(anthology_df):,} papers")

# Drop papers with missing title or abstract
n_before = len(anthology_df)
anthology_df = anthology_df.dropna(subset=["title", "abstract"])
anthology_df = anthology_df[
    (anthology_df["title"].str.strip() != "")
    & (anthology_df["abstract"].str.strip() != "")
]
print(f"Dropped {n_before - len(anthology_df):,} papers with missing title/abstract")
print(f"Remaining: {len(anthology_df):,} papers")

anthology_df[["bibkey", "title", "year"]].head()

Anthology corpus: 61,486 papers
Dropped 2,273 papers with missing title/abstract
Remaining: 59,213 papers


,bibkey,title,year
1,cettolo-etal-2013-report,Report on the 10th IWSLT evaluation campaign,2013
2,lo-wu-2013-human,Human semantic MT evaluation with HMEANT for I...,2013
3,birch-etal-2013-english,English SLT and MT system description for the ...,2013
4,aue-etal-2013-msr,MSR-FBK IWSLT 2013 SLT system description,2013
5,lo-etal-2013-improving-machine,Improving machine translation into Chinese by ...,2013


## 2. RegEx Bucketing

Each paper is scored against hand-crafted regex patterns that capture lexical signals for benchmark introduction and text classification relevance. Signals are grouped into categories (e.g. `introduces`, `evaluates_on`, `classif_tasks`, `known_classif_benchmarks`) and aggregated into composite scores.

Papers are then assigned to one of three buckets:
- **Bucket A** — Strong evidence the paper introduces a text-classification benchmark (high-confidence positive).
- **Bucket B** — Partial or ambiguous evidence; requires LLM adjudication.
- **Bucket C** — No relevant signal (likely negative).

A post-filter detects papers that *introduce a method* (not a benchmark) to reduce false positives in Bucket A.

In [5]:
from src.labelling.regex_buckets import assign_bucket, compute_scores

# Score every paper against regex signal categories
score_cols = anthology_df.apply(
    lambda r: compute_scores(r["title"], r["abstract"]), axis=1
)
score_df = pd.DataFrame(score_cols.tolist())

anthology_bucket_df = pd.concat(
    [anthology_df.reset_index(drop=True), score_df.reset_index(drop=True)],
    axis=1,
)
anthology_bucket_df["bucket"] = anthology_bucket_df.apply(assign_bucket, axis=1)

In [6]:
bucket_counts = anthology_bucket_df["bucket"].value_counts().sort_index()
print("Bucket distribution:")
print(bucket_counts.to_string())
print(f"\nTotal: {len(anthology_bucket_df):,} papers")

Bucket distribution:
bucket
A      719
B     7840
C    50654

Total: 59,213 papers


In [7]:
anthology_bucket_df.to_parquet(
    DATA_DIR / "labeled" / "anthology_enriched_with_bucket_test.parquet"
)

## 3. Stratified Sampling

We draw three non-overlapping splits from the bucketed corpus, each stratified by bucket and year to ensure temporal representativeness:

| Split | Purpose | Size per bucket |
|-------|---------|-----------------|
| **Gold** | Human-evaluated held-out test set | 70 / bucket |
| **Train** | LLM-labelled training data | A=500, B=400, C=300 |
| **Val** | Validation (15% of train pool) | ~15% of train per bucket |

In [ ]:
from src.labelling.regex_buckets import stratified_bucket_sample

# Gold set — held out for human evaluation
gold = stratified_bucket_sample(
    anthology_bucket_df,
    n_a=GOLD_PER_BUCKET,
    n_b=GOLD_PER_BUCKET,
    n_c=GOLD_PER_BUCKET,
    seed=SEED,
)
remaining = anthology_bucket_df.drop(gold.index)

# Train + Val pool
train_val = stratified_bucket_sample(
    remaining,
    n_a=TRAIN_N_A,
    n_b=TRAIN_N_B,
    n_c=TRAIN_N_C,
    seed=SEED + 1,
)

# Split train/val within each bucket
val_idx = (
    train_val.groupby("bucket", group_keys=False)
    .apply(lambda x: x.sample(frac=VAL_FRAC, random_state=SEED))
    .index
)
val = train_val.loc[val_idx]
train = train_val.drop(val_idx)

In [ ]:
summary = (
    pd.DataFrame(
        {
            "Gold": gold["bucket"].value_counts(),
            "Train": train["bucket"].value_counts(),
            "Val": val["bucket"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)
summary.loc["Total"] = summary.sum()
summary

In [ ]:
splits_dir = DATA_DIR / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

gold.to_parquet(splits_dir / "gold.parquet", index=False)
train.to_parquet(splits_dir / "train.parquet", index=False)
val.to_parquet(splits_dir / "val.parquet", index=False)

## 4. LLM Labelling

Each paper in the training split is classified independently by two LLMs via their respective Batch APIs:
- **Mistral Small** (`mistral-small-latest`) — structured output via JSON schema.
- **Claude Haiku** (`claude-haiku-4-5-20251001`) — structured output via tool use.

Both models receive the same system prompt that defines POSITIVE / NEGATIVE / UNSURE labels (see `src/labelling/llm_labeller.py` for the full prompt and Pydantic schema).

> **Note:** Batch API calls are non-deterministic and cost-incurring. The cells below load pre-computed results. To re-run the batch jobs from scratch, uncomment the submission cells.

In [ ]:
from src.labelling.llm_labeller import (
    mistral_batch_submit,
    mistral_batch_results,
    claude_batch_submit,
    claude_batch_results,
)

### 4.1 Submit batch jobs (optional)

Uncomment the cells below to submit new batch jobs. Each call returns a job ID that can be used to retrieve results once processing completes.

In [ ]:
# job_id_mistral = mistral_batch_submit(train)
# job_id_mistral

In [ ]:
# job_id_claude = claude_batch_submit(train)
# job_id_claude

### 4.2 Load pre-computed results

In [ ]:
result_mistral = pd.read_parquet(DATA_DIR / "splits" / "train_mistral_1.parquet")
result_claude = pd.read_parquet(DATA_DIR / "splits" / "train_claude_1.parquet")

print(f"Mistral labels: {len(result_mistral):,} papers")
print(result_mistral["mistral_label"].value_counts().to_string())
print(f"\nClaude labels:  {len(result_claude):,} papers")
print(result_claude["claude_label"].value_counts().to_string())

## 5. Inter-Annotator Agreement

We measure agreement between the two LLM annotators using Cohen's $\kappa$, which corrects for chance agreement. Interpretation thresholds (Landis & Koch, 1977):

| $\kappa$ | Interpretation |
|-----------|----------------|
| < 0.20 | Slight |
| 0.21–0.40 | Fair |
| 0.41–0.60 | Moderate |
| 0.61–0.80 | Substantial |
| 0.81–1.00 | Almost perfect |

In [ ]:
# Align on shared bibkeys
merged = result_mistral[["bibkey", "mistral_label"]].merge(
    result_claude[["bibkey", "claude_label"]],
    on="bibkey",
    how="inner",
)
print(f"Papers with both labels: {len(merged):,}")

kappa = cohen_kappa_score(merged["mistral_label"], merged["claude_label"])
print(f"Cohen's kappa: {kappa:.3f}")

In [ ]:
labels = sorted(merged["mistral_label"].unique())
cm = confusion_matrix(merged["mistral_label"], merged["claude_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = "Mistral"
cm_df.columns.name = "Claude"
cm_df

### Disagreement analysis

Papers where the two annotators disagree are the most informative cases for understanding model biases and refining the classification prompt.

In [ ]:
disagreements = merged[merged["mistral_label"] != merged["claude_label"]]
print(
    f"Disagreements: {len(disagreements)} / {len(merged)} ({len(disagreements) / len(merged):.1%})"
)

# Show disagreement breakdown
pd.crosstab(
    disagreements["mistral_label"],
    disagreements["claude_label"],
    margins=True,
)

## 6. Export trainer-ready splits

The classifier expects a Dataset with three text columns: `title` (str), `abstract` (str), and `label` (int, binary). We use Claude's labels as the reference annotator, drop UNSURE samples, and map POSITIVE → 1, NEGATIVE → 0.

In [3]:
LABEL_MAP = {"POSITIVE": 1, "NEGATIVE": 0}


def prepare_split(df: pd.DataFrame, label_col: str) -> pd.DataFrame:
    """Filter UNSURE, drop missing abstracts/titles, map labels to int."""
    out = df[df[label_col].isin(LABEL_MAP)].copy()
    out = out.dropna(subset=["abstract", "title"])
    out = out[(out["abstract"].str.strip() != "") & (out["title"].str.strip() != "")]
    out["label"] = out[label_col].map(LABEL_MAP)
    return out[["bibkey", "title", "abstract", "label"]]

In [4]:
label_col = "claude_label"

train_ready = prepare_split(
    pd.read_parquet(DATA_DIR / "splits" / "train_claude_1.parquet"), label_col
)
val_ready = prepare_split(
    pd.read_parquet(DATA_DIR / "splits" / "val_claude_1.parquet"), label_col
)
gold_ready = prepare_split(
    pd.read_parquet(DATA_DIR / "splits" / "gold_claude_1.parquet"), label_col
)

for name, df in [("Train", train_ready), ("Val", val_ready), ("Gold", gold_ready)]:
    n_pos = (df["label"] == 1).sum()
    n_neg = (df["label"] == 0).sum()
    print(f"{name:5s}: {len(df):,} papers  (pos={n_pos}, neg={n_neg})")

Train: 999 papers  (pos=306, neg=693)
Val  : 179 papers  (pos=69, neg=110)
Gold : 206 papers  (pos=48, neg=158)


In [ ]:
ready_dir = DATA_DIR / "splits" / "ready"
ready_dir.mkdir(parents=True, exist_ok=True)

train_ready.to_parquet(ready_dir / "train.parquet", index=False)
val_ready.to_parquet(ready_dir / "val.parquet", index=False)
gold_ready.to_parquet(ready_dir / "test.parquet", index=False)

Saved to ..\data\splits\ready


## 7. Additional Sampling (no overlap)

Sample more papers for LLM annotation while excluding all previously annotated bibkeys (gold, train, val from round 1). Uses the same `stratified_bucket_sample` method on the remaining pool.

In [5]:
# Collect all bibkeys already annotated in round 1
existing_gold = pd.read_parquet(DATA_DIR / "splits" / "gold.parquet")
existing_train = pd.read_parquet(DATA_DIR / "splits" / "train.parquet")
existing_val = pd.read_parquet(DATA_DIR / "splits" / "val.parquet")

anthology_bucket_df = pd.read_parquet(
    DATA_DIR / "labeled" / "anthology_enriched_with_bucket.parquet"
)

annotated_bibkeys = set(
    pd.concat(
        [existing_gold["bibkey"], existing_train["bibkey"], existing_val["bibkey"]]
    )
)
print(f"Already annotated: {len(annotated_bibkeys):,} papers")

# Remove already-annotated papers from the bucketed corpus
pool = anthology_bucket_df[~anthology_bucket_df["bibkey"].isin(annotated_bibkeys)]
print(f"Remaining pool: {len(pool):,} papers")
print(pool["bucket"].value_counts().sort_index().to_string())

Already annotated: 1,410 papers
Remaining pool: 60,076 papers
bucket
A      150
B     7381
C    52545


In [ ]:
# --- Adjust these for round 2 ---
TRAIN2_N_A, TRAIN2_N_B, TRAIN2_N_C = 500, 400, 300
VAL2_FRAC = 0.15
SEED2 = 123  # Different seed for fresh sampling

# Sample new train+val from the pool (same stratified method)
train_val_2 = stratified_bucket_sample(
    pool,
    n_a=TRAIN2_N_A,
    n_b=TRAIN2_N_B,
    n_c=TRAIN2_N_C,
    seed=SEED2,
)

# Split into train / val
val2_idx = (
    train_val_2.groupby("bucket", group_keys=False)
    .apply(lambda x: x.sample(frac=VAL2_FRAC, random_state=SEED2))
    .index
)
val_2 = train_val_2.loc[val2_idx]
train_2 = train_val_2.drop(val2_idx)

# Sanity check: no overlap
assert train_2["bibkey"].isin(annotated_bibkeys).sum() == 0, (
    "Overlap detected in train!"
)
assert val_2["bibkey"].isin(annotated_bibkeys).sum() == 0, "Overlap detected in val!"

summary2 = (
    pd.DataFrame(
        {
            "Train_r2": train_2["bucket"].value_counts(),
            "Val_r2": val_2["bucket"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
    .sort_index()
)
summary2.loc["Total"] = summary2.sum()
summary2

In [ ]:
# Save round 2 splits
train_2.to_parquet(splits_dir / "train_r2.parquet", index=False)
val_2.to_parquet(splits_dir / "val_r2.parquet", index=False)
print(f"Saved {len(train_2):,} train + {len(val_2):,} val papers to {splits_dir}")